In [2]:
import json
import pandas as pd
from collections import Counter

fichiers = {
    "Hugo": "vldbench_batch_01 (1).json",
    "Eugene": "vldbench_batch_01 (2).json",
    " Kwabena": "vldbench_batch_01 (3).json",
    "Amadou": "vldbench_batch_01 (4).json",
    "Annotateur 5": "vldbench_batch_01 (5).json"
}

# EXTRACTIONS DES DONNEES DE DESACCORDS 
Les données sont enreaigistrées dans un fichire .CSV  rapport_desaccords_equipe
 les données avec les valeurs sont extraites pour les valeurs divergentes 

In [ ]:
donnees_fusionnees = {}

print("Chargement des fichiers et alignement des données...")

for nom_annotateur, chemin_fichier in fichiers.items():
    try:
        with open(chemin_fichier, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            for ancre in data:
                id_brut = ancre.get("metadata", {}).get("id", "Inconnu")
                id_ancre = f"ID {id_brut}" 
                
                titre_ancre = ancre.get("news", "Titre inconnu")
                
                if 'database' in ancre:
                    for index, cible in enumerate(ancre['database']):
                        relation = cible.get('related')
                        
                        # Si l'article a été annoté
                        if relation is not None:
                            pos_cible = f"T{index + 1}"
                            cle_unique = (id_ancre, pos_cible)
                            
                            if cle_unique not in donnees_fusionnees:
                                donnees_fusionnees[cle_unique] = {}
                                
                            # On normalise le texte (majuscule au début) pour éviter les faux désaccords
                            donnees_fusionnees[cle_unique][nom_annotateur] = str(relation).title().strip()
                            
    except FileNotFoundError:
        print(f" Le fichier '{chemin_fichier}' est introuvable.")

# 3. FILTRAGE DES DÉSACCORDS ET CRÉATION DU TABLEAU
lignes_desaccords = []

for (id_ancre, pos_cible), annotations in donnees_fusionnees.items():
    # On récupère toutes les réponses données pour cet article (en ignorant les vides)
    reponses_uniques = set(annotations.values())
    
    # S'il y a plus d'une réponse différente, c'est qu'il y a un désaccord !
    if len(reponses_uniques) > 1:
        ligne = {
            "ID Ancre": id_ancre,
            "Cible": pos_cible
        }
        
        # On ajoute la réponse de chaque annotateur dans la ligne
        for nom_annotateur in fichiers.keys():
            ligne[nom_annotateur] = annotations.get(nom_annotateur, "Non annoté")
            
        lignes_desaccords.append(ligne)

# 4. AFFICHAGE ET SAUVEGARDE DU TABLEAU DE DÉSACCORDS
if lignes_desaccords:
    df_desaccords = pd.DataFrame(lignes_desaccords)
    
    
    print(f" {len(lignes_desaccords)} DÉSACCORDS TROUVÉS (Relations non identiques)")
  
    
    # Affiche le tableau dans la console
    print(df_desaccords.to_string(index=False))
    
    # Sauvegarde en fichier CSV pour Excel
    nom_fichier_sortie = "rapport_desaccords_equipe.csv"
    df_desaccords.to_csv(nom_fichier_sortie, index=False, encoding='utf-8')
    print(f"\n Le tableau a été sauvegardé avec succès dans le fichier : {nom_fichier_sortie}")

else:
    print("\nParfait ! Aucun désaccord de relation trouvé ou les fichiers n'ont pas de données communes.")

Chargement des fichiers et alignement des données...
 Le fichier 'vldbench_batch_01 (5).json' est introuvable.
 330 DÉSACCORDS TROUVÉS (Relations non identiques)
ID Ancre Cible         Hugo       Eugene      Kwabena       Amadou Annotateur 5
    ID 0    T1 Undetermined   Supporting Undetermined Undetermined   Non annoté
    ID 0    T2 Undetermined   Supporting Undetermined Undetermined   Non annoté
    ID 0    T6  Not_Related Undetermined  Not_Related  Not_Related   Non annoté
    ID 1    T2 Undetermined   Supporting   Supporting Undetermined   Non annoté
    ID 1    T6  Not_Related Undetermined  Not_Related  Not_Related   Non annoté
    ID 2    T1   Supporting   Supporting   Supporting Undetermined   Non annoté
    ID 2    T6  Not_Related Undetermined  Not_Related Undetermined   Non annoté
    ID 3    T2 Undetermined   Supporting Undetermined      Against   Non annoté
    ID 3    T6  Not_Related Undetermined  Not_Related Undetermined   Non annoté
    ID 4    T1 Undetermined   Supporti

In [ ]:

# 3. FILTRAGE : LES CONFLITS DE MAJORITÉ

lignes_majorite = []
# Dans votre version du code, 'infos' contient DIRECTEMENT les votes
for (id_ancre, pos_cible), infos in donnees_fusionnees.items():
    
    # On filtre pour ignorer les "Non annoté" dans le calcul
    votes_valides = {annotateur: choix for annotateur, choix in infos.items() 
                     if choix not in ["Non annoté", "None", None]}
    
    reponses = list(votes_valides.values())
    total_votants = len(reponses)
    
    # S'il y a moins de 2 votants valides, pas de conflit de majorité possible
    if total_votants < 2:
        continue
        
    compteur = Counter(reponses)
    vote_majoritaire, compte_majorite = compteur.most_common(1)[0]
    
    # CONDITION : On a une majorité stricte (> moitié) MAIS PAS l'unanimité
    # Exemple: 3 sur 4, ou 4 sur 5.
    if compte_majorite > (total_votants / 2) and compte_majorite < total_votants:
        
        pourcentage = (compte_majorite / total_votants) * 100
        
        # On cherche qui n'a pas suivi la majorité
        details_minorite = []
        for annotateur, choix in votes_valides.items():
            if choix != vote_majoritaire:
                details_minorite.append(f"{annotateur} a mis '{choix}'")
        
        ligne = {
            "ID Ancrage": id_ancre,
            "Cible": pos_cible,
            "Vote Majoritaire": vote_majoritaire,
            "Majorité (%)": f"{pourcentage:.0f}% ({compte_majorite}/{total_votants})",
            "Détail Minorité": " | ".join(details_minorite)
        }
    
        lignes_majorite.append(ligne)
# 4. AFFICHAGE ET SAUVEGARDE DU TABLEAU DE MAJORITÉ
"""if lignes_majorite:
    df_majorite = pd.DataFrame(lignes_majorite)
    df_majorite = df_majorite.sort_values(by=["ID Ancrage", "Cible"])
    
    print(f" {len(lignes_majorite)} CONFLITS DE MAJORITÉ TROUVÉS (ex: 3/4 ou 4/5 d'accord)")
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_majorite.to_string(index=False))
    
    # Sauvegarde
    nom_fichier = "analyse_majorite_groupe.csv"
    df_majorite.to_csv(nom_fichier, index=False, encoding='utf-8-sig')
    print(f"\n Tableau sauvegardé sous : '{nom_fichier}'")
else:
    print("\nAucun conflit de majorité trouvé.")"""

'if lignes_majorite:\n    df_majorite = pd.DataFrame(lignes_majorite)\n    df_majorite = df_majorite.sort_values(by=["ID Ancrage", "Cible"])\n\n    print(f" {len(lignes_majorite)} CONFLITS DE MAJORITÉ TROUVÉS (ex: 3/4 ou 4/5 d\'accord)")\n\n    pd.set_option(\'display.max_columns\', None)\n    pd.set_option(\'display.width\', 1000)\n    print(df_majorite.to_string(index=False))\n\n    # Sauvegarde\n    nom_fichier = "analyse_majorite_groupe.csv"\n    df_majorite.to_csv(nom_fichier, index=False, encoding=\'utf-8-sig\')\n    print(f"\n Tableau sauvegardé sous : \'{nom_fichier}\'")\nelse:\n    print("\nAucun conflit de majorité trouvé.")'

# FONCTION : STATISTIQUES INDIVIDUELLES 
Les fonction '

In [5]:

# FONCTION : STATISTIQUES INDIVIDUELLES 
def calculer_statistiques_annotateurs(donnees_fusionnees, liste_annotateurs):
    
    stats = {annotateur: {"Total": 0, "Accords": 0, "Désaccords": 0, "Cas_Majorite_Valides": 0} 
             for annotateur in liste_annotateurs}
    
    for cle, infos in donnees_fusionnees.items():
        
        # On récupère uniquement les vrais votes
        votes_valides = {annot: choix for annot, choix in infos.items() 
                         if annot in liste_annotateurs and choix not in ["Non annoté", "None", None, ""]}
        
        total_votants = len(votes_valides)
        
        # On compte le total de travail de chacun
        for annotateur in votes_valides.keys():
            stats[annotateur]["Total"] += 1
            
        # NOUVELLE RÈGLE : On ne cherche une majorité que si au moins 3 personnes ont voté !
        if total_votants >= 3:
            compteur = Counter(votes_valides.values())
            vote_majoritaire, compte_majorite = compteur.most_common(1)[0]
            
            # Si on a une vraie majorité (ex: 2/3, 3/4, 3/5, 4/5)
            if compte_majorite > (total_votants / 2):
                for annotateur, choix in votes_valides.items():
                    stats[annotateur]["Cas_Majorite_Valides"] += 1 # On note que la personne a participé à un vote avec majorité
                    
                    if choix == vote_majoritaire:
                        stats[annotateur]["Accords"] += 1
                    else:
                        stats[annotateur]["Désaccords"] += 1

    # Formatage du tableau de résultats
    lignes_resultats = []
    for annotateur, donnees in stats.items():
        total_global = donnees["Total"]
        total_valable = donnees["Cas_Majorite_Valides"]
        accords = donnees["Accords"]
        desaccords = donnees["Désaccords"]
        
        if total_global > 0:
            # Le score se calcule uniquement sur les cas où une majorité existait
            if total_valable > 0:
                score_accord_pct = (accords / total_valable) * 100
                score_desaccord_pct = (desaccords / total_valable) * 100
            else:
                score_accord_pct = 0
                score_desaccord_pct = 0
                
            lignes_resultats.append({
                "Annotateur": annotateur,
                "Total Annotations": total_global,
                "Score d'Accord (Suit la majorité)": f"{accords} fois ({score_accord_pct:.1f}%)",
                "A contredit la majorité": f"{desaccords} fois",
                "Taux de désaccord": f"{score_desaccord_pct:.1f}%"
            })
            
    df_stats = pd.DataFrame(lignes_resultats)
    df_stats = df_stats.sort_values(by="Taux de désaccord", ascending=False) # Trie du plus grand "rebelle" au plus sage
    
    return df_stats

 5. STATISTIQUES AVANCÉES : ANALYSE DYNAMIQUE PAR PHASES
 


In [ ]:

# 5. STATISTIQUES AVANCÉES : ANALYSE DYNAMIQUE PAR PHASES

stats_par_phase = {}
paires_en_conflit_2_personnes = Counter() 

for cle, infos in donnees_fusionnees.items():
    # On récupère les votes valides
    votes = {annot: choix for annot, choix in infos.items() 
             if choix not in ["Non annoté", "None", None, ""]}
    
    n_votants = len(votes)
    
 
    if n_votants >= 3:
        # Si cette phase (ex: 5 votants) n'existe pas encore, on la crée automatiquement
        if n_votants not in stats_par_phase:
            stats_par_phase[n_votants] = {annot: {"Total": 0, "Désaccords": 0} for annot in fichiers.keys()}
            
        compteur = Counter(votes.values())
        vote_majoritaire, compte_majorite = compteur.most_common(1)[0]
        
        # On ajoute +1 au total de chaque personne présente
        for annot in votes:
            stats_par_phase[n_votants][annot]["Total"] += 1
            
        # S'il y a une majorité stricte (> moitié)
        if compte_majorite > (n_votants / 2):
            for annot, choix in votes.items():
                if choix != vote_majoritaire:
                    stats_par_phase[n_votants][annot]["Désaccords"] += 1
    elif n_votants == 2:
        choix_uniques = set(votes.values())
        
        # S'ils ont mis deux choses différentes
        if len(choix_uniques) == 2:
            annotateurs_presents = list(votes.keys())
            paire = " et ".join(sorted(annotateurs_presents))
            paires_en_conflit_2_personnes[paire] += 1



# 6. AFFICHAGE DYNAMIQUE DU RAPPORT


def afficher_tableau_phase(dico_stats, titre):
    lignes = []
    for annot, data in dico_stats.items():
        if data["Total"] > 0:
            pct = (data["Désaccords"] / data["Total"]) * 100
            lignes.append({
                "Annotateur": annot,
                "Total Annoté": data["Total"],
                "A contredit la majorité": f"{data['Désaccords']} fois",
                "Taux de désaccord": f"{pct:.1f}%"
            })
    
    if lignes:
        df = pd.DataFrame(lignes).sort_values(by="Taux de désaccord", ascending=False)
        print(f" {titre}")
        print(df.to_string(index=False))

print("  RAPPORT DÉTAILLÉ DE L'ÉVOLUTION DU GROUPE")


# On affiche les phases automatiquement, de la plus grande à la plus petite (ex: 10, puis 9, etc.)
for phase in sorted(stats_par_phase.keys(), reverse=True):
    # Calcul dynamique du seuil de majorité pour l'affichage
    seuil_majorite = (phase / 2) + 0.1
    titre_phase = f"PHASE OÙ {phase} PERSONNES ONT VOTÉ (Majorité > {phase/2:.1f})"
    afficher_tableau_phase(stats_par_phase[phase], titre_phase)

# Affichage des conflits à 2
print("\n" + "-"*60)
print("PHASE OÙ SEULEMENT 2 PERSONNES ONT VOTÉ (Contradictions directes)")
print("-"*60)
if not paires_en_conflit_2_personnes:
    print("Aucune contradiction directe à 2 personnes.")
else:
    for paire, nb_conflits in paires_en_conflit_2_personnes.most_common():
        print(f"  {paire} se sont contredits {nb_conflits} fois.")
print("\n")

  RAPPORT DÉTAILLÉ DE L'ÉVOLUTION DU GROUPE
 PHASE OÙ 4 PERSONNES ONT VOTÉ (Majorité > 2.0)
Annotateur  Total Annoté A contredit la majorité Taux de désaccord
      Hugo           120                  8 fois              6.7%
    Amadou           120                  8 fois              6.7%
    Eugene           120                 25 fois             20.8%
   Kwabena           120                  1 fois              0.8%
 PHASE OÙ 3 PERSONNES ONT VOTÉ (Majorité > 1.5)
Annotateur  Total Annoté A contredit la majorité Taux de désaccord
   Kwabena           300                 23 fois              7.7%
    Amadou           300                 16 fois              5.3%
      Hugo           300                 89 fois             29.7%

------------------------------------------------------------
PHASE OÙ SEULEMENT 2 PERSONNES ONT VOTÉ (Contradictions directes)
------------------------------------------------------------
  Amadou et Hugo se sont contredits 137 fois.


